In [3]:
import json
import operator
from pathlib import Path
from typing import TypedDict, Annotated, List, Dict, Any

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import (
    AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage
)
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# --- CARICAMENTO DATI DAI FILE JSON ALLEGATI ---
DATA_DIR = Path("data")

with open(DATA_DIR / "claims.json", "r", encoding="utf-8") as f:
    CLAIMS_DB = {c["claim_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "customers.json", "r", encoding="utf-8") as f:
    CUSTOMERS_DB = {c["customer_id"]: c for c in json.load(f)} #

with open(DATA_DIR / "policies.json", "r", encoding="utf-8") as f:
    POLICIES_DB = {p["policy_id"]: p for p in json.load(f)} #

with open(DATA_DIR / "requirements.json", "r", encoding="utf-8") as f:
    REQUIREMENTS_DB = {r["loss_type"]: r for r in json.load(f)} #

llm = ChatOllama(model="qwen3.5:9b", temperature=0)


# ==============================================================================
# 2. DEFINIZIONE DEI TOOL PER LE 6 TASK
# ==============================================================================

# --- Gestione errori (stesso principio di take_action nei notebook 02/03): ---
# un tool non solleva mai eccezioni verso il grafo, restituisce SEMPRE una stringa
# "ERRORE — ..." che il LLM legge e può spiegare all'utente.
def _errore_tool(nome: str, e: Exception) -> str:
    """Eccezione imprevista dentro un tool (es. dati JSON malformati)."""
    print(f"  ⚠️  [{nome}] {type(e).__name__}: {e}")
    return f"ERRORE — {nome}: {type(e).__name__}: {e}"


def _errore_validazione(e) -> str:
    """Argomenti con tipo sbagliato (es. stringa al posto di una lista): sono
    rifiutati da LangChain PRIMA di entrare nella funzione, quindi un try/except
    nel corpo non li vedrebbe. Vengono gestiti con handle_validation_error."""
    try:
        dettagli = "; ".join(
            f"{'.'.join(str(p) for p in err['loc'])}: {err['msg']}" for err in e.errors()
        )
    except Exception:
        dettagli = str(e)
    return f"ERRORE — Parametri non validi per il tool ({dettagli}). Riprova con i tipi corretti."


# Stringa ESATTA che l'orchestrator si aspetta quando serve il perito.
# Il router del collega può confrontarla con "==" per instradare verso il nodo
# "chiedi al perito" (che sceglie il perito e restituisce sì/no).
CHIEDI_PERITO = "chiedi a un perito"


def valuta_importo_danno(estimated_damage: float, limite_massimo: float, franchigia: float = 0) -> str:
    """Valuta l'importo del danno rispetto al limite massimo stabilito.

    Questa funzione NON sceglie il perito e NON prende la decisione finale:
    quando serve un perito si limita a segnalarlo, il resto spetta al nodo
    dell'orchestrator (che risponde sì/no).

    Args:
        estimated_damage: Danno stimato in euro.
        limite_massimo: Limite massimo stabilito (in euro). Il danno lo "supera"
            solo se è strettamente maggiore: un danno uguale al limite NON lo supera.
        franchigia: Soglia minima in euro (default 0 = nessuna franchigia).
            Un danno minore o uguale alla franchigia non viene risarcito.

    Returns:
        - "chiedi a un perito" (esattamente questa stringa) se il danno supera il limite massimo;
        - "RIFIUTATO — ..." se il danno è sotto o pari alla franchigia;
        - "OK — ..." se il danno rientra tra franchigia e limite massimo (flusso normale);
        - "ERRORE — ..." se i dati in ingresso non sono validi (la funzione non solleva mai eccezioni).
    """
    # --- 1. Validazione input: restituisce una stringa invece di sollevare eccezioni ---
    try:
        danno = float(estimated_damage)
        limite = float(limite_massimo)
        franch = float(franchigia)
    except (TypeError, ValueError):
        return "ERRORE — danno, limite massimo e franchigia devono essere numeri."

    # "not (x >= 0)" intercetta sia i valori negativi sia NaN
    if not (danno >= 0) or not (limite >= 0) or not (franch >= 0):
        return "ERRORE — danno, limite massimo e franchigia devono essere numeri >= 0."

    # --- 2. Caso richiesto: il danno SUPERA il limite massimo -> perito ---
    # Si controlla per primo: oltre il limite la decisione non spetta a questa funzione.
    if danno > limite:
        return CHIEDI_PERITO

    # --- 3. Flusso normale: il danno NON supera il limite massimo ---
    # Sotto o pari alla franchigia: nessun risarcimento.
    if danno <= franch:
        return f"RIFIUTATO — Danno stimato ({danno}€) inferiore o pari alla franchigia ({franch}€)."

    # Tra franchigia e limite massimo: approvabile.
    return f"OK — Danno stimato ({danno}€) compreso tra franchigia ({franch}€) e limite massimo ({limite}€)."


@tool
def verifica_copertura_e_tipo_polizza(policy_id: str, loss_type: str) -> str:
    """Task 3 e Task 4: Verifica se la polizza è attiva, se copre il tipo di evento (Task 3)

    e restituisce il tipo di assicurazione e prodotto (Task 4).

    Args:
        policy_id: Codice della polizza
        loss_type: Tipo di evento/danno del sinistro
    """
    try:
        if not policy_id or not loss_type:
            return "ERRORE — policy_id e loss_type sono obbligatori."

        policy = POLICIES_DB.get(policy_id) #
        if not policy:
            return f"RIFIUTATO — Polizza '{policy_id}' non trovata a sistema." #

        if not policy.get("active", False): #
            return f"RIFIUTATO — La polizza '{policy_id}' risulta DISATTIVA." #

        coverage = policy.get("coverage", "") #
        if coverage != loss_type: #
            return f"RIFIUTATO — Il tipo evento '{loss_type}' non è coperto dal tipo garanzia '{coverage}'." #

        prodotto = policy.get("product", "Sconosciuto") #
        return f"OK — Polizza attiva. Prodotto: '{prodotto}', Copertura: '{coverage}'." #
    except Exception as e:
        return _errore_tool("verifica_copertura_e_tipo_polizza", e)


@tool
def analizza_documenti(claim_documents: List[str], loss_type: str) -> str:
    """Task 1 e Task 6: Ispeziona i documenti forniti dal cliente (Task 6)

    e verifica se tutti i documenti richiesti sono presenti (Task 1).

    Args:
        claim_documents: Lista dei documenti inviati dal cliente per il sinistro
        loss_type: Tipo di evento/danno del sinistro, usato per cercare i documenti richiesti
    """
    try:
        if not loss_type:
            return "ERRORE — loss_type è obbligatorio."

        req = REQUIREMENTS_DB.get(loss_type) #
        if not req:
            return f"ERRORE — Tipo evento '{loss_type}' non presente nei requisiti." #

        req_docs = set(req.get("required_documents", [])) #
        provided_docs = set(claim_documents) #

        mancanti = list(req_docs - provided_docs) #

        if mancanti:
            return f"INCOMPLETO — Documenti forniti: {list(provided_docs)}. Documenti mancanti: {mancanti}." #

        return f"OK — Documentazione completa. Documenti forniti: {list(provided_docs)}." #
    except Exception as e:
        return _errore_tool("analizza_documenti", e)


@tool
def calcola_budget_e_limiti(estimated_damage: float, policy_id: str, loss_type: str) -> str:
    """Task 2 e Task 5: Verifica franchigia (tetto minimo) e massimale (tetto massimo) (Task 5),

    e controlla il rientro nel budget di approvazione automatica (Task 2).

    Args:
        estimated_damage: Danno stimato in euro
        policy_id: Codice della polizza
        loss_type: Tipo di evento/danno del sinistro
    """
    try:
        # "not (x >= 0)" intercetta sia i valori negativi sia NaN
        if not (estimated_damage >= 0):
            return f"ERRORE — Danno stimato non valido ({estimated_damage}): deve essere un numero >= 0."
        if not policy_id or not loss_type:
            return "ERRORE — policy_id e loss_type sono obbligatori."

        policy = POLICIES_DB.get(policy_id) #
        req = REQUIREMENTS_DB.get(loss_type) #

        if not policy or not req:
            return "ERRORE — Dati polizza o requisiti mancanti." #

        deductible = policy.get("deductible", 0) #
        max_claim = policy.get("max_claim", 0) #
        auto_max = req.get("automatic_review_max", 0) #

        # Check 1: Sotto la franchigia (tetto minimo)
        if estimated_damage <= deductible: #
            return f"RIFIUTATO — Danno stimato ({estimated_damage}€) inferiore o pari alla franchigia/tetto minimo ({deductible}€)." #

        # Check 2: Supera il massimale (tetto massimo) -> serve il perito:
        # la stringa esatta "chiedi a un perito" viene poi gestita dall'orchestrator
        esito_limite = valuta_importo_danno(estimated_damage, max_claim, deductible) #
        if esito_limite == CHIEDI_PERITO or esito_limite.startswith("ERRORE"): #
            return esito_limite #

        # Check 3: Rientro nel budget per approvazione automatica
        if estimated_damage > auto_max: #
            return f"ESCALATION — Danno stimato ({estimated_damage}€) supera la soglia di approvazione automatica ({auto_max}€). Richiesta revisione manuale." #

        return f"OK — Danno stimato ({estimated_damage}€) compreso tra franchigia ({deductible}€) e massimale ({max_claim}€), ed entro la soglia automatica ({auto_max}€)." #
    except Exception as e:
        return _errore_tool("calcola_budget_e_limiti", e)


@tool
def get_info_sinistro(claim_id: str) -> str:
    """Estrae i dati di sintesi di un sinistro (policy_id, loss_type, estimated_damage, documents).

    Args:
        claim_id: Codice del sinistro (es. 'CL001')
    """
    try:
        if not claim_id or not claim_id.strip():
            return "ERRORE — claim_id mancante o vuoto."

        claim = CLAIMS_DB.get(claim_id)
        if not claim:
            return f"ERRORE — Sinistro '{claim_id}' non trovato."

        return (
            f"Dati Sinistro '{claim_id}':\n"
            f"- Policy ID: {claim.get('policy_id')}\n"
            f"- Tipo Danno: {claim.get('loss_type')}\n"
            f"- Danno Stimato: {claim.get('estimated_damage')} EUR\n"
            f"- Documenti Inviati: {claim.get('documents', [])}"
        )
    except Exception as e:
        return _errore_tool("get_info_sinistro", e)


tools_assicurativi = [
    get_info_sinistro,
    verifica_copertura_e_tipo_polizza,
    analizza_documenti,
    calcola_budget_e_limiti,
]

# Errori di validazione degli argomenti -> stringa "ERRORE — ..." invece di eccezione
for _t in tools_assicurativi:
    _t.handle_validation_error = _errore_validazione


# ==============================================================================
# 3. PATTERN AGENT: CLASSE REACT RIUTILIZZABILE (LANGGRAPH)
# ==============================================================================

class Agent:
    """Agente ReAct con supporto a StateGraph e MemorySaver."""

    def __init__(self, model, tools, checkpointer=None, system=""):
        self.system = system

        class AgentState(TypedDict):
            messages: Annotated[list[AnyMessage], operator.add]

        self.AgentState = AgentState

        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)

        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state):
        return len(state["messages"][-1].tool_calls) > 0

    def call_llm(self, state):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {"messages": [message]}

    def take_action(self, state):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"  🔧 Chiamata tool: {t['name']}({t['args']})")
            if t["name"] not in self.tools:
                result = f"Tool '{t['name']}' non esiste. Disponibili: {list(self.tools.keys())}"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(
                tool_call_id=t["id"],
                name=t["name"],
                content=str(result)
            ))
        return {"messages": results}



# ==============================================================================
# 4. ISTANZIAZIONE DELL'AGENTE E SYSTEM PROMPT
# ==============================================================================

system_prompt = """Sei l'Assistente Pratiche Assicurative.
Processa la richiesta di valutazione di un sinistro seguendo rigorosamente questi passi:

1. Chiama `get_info_sinistro` usando il claim_id fornito per ottenere le informazioni di base.
2. Chiama `verifica_copertura_e_tipo_polizza` per accertarti che la polizza sia attiva e che copra il danno.
3. Chiama `analizza_documenti` per verificare se la documentazione inviata è completa.
4. Chiama `calcola_budget_e_limiti` per verificare che l'importo rientri tra franchigia, massimale e soglia automatica.
5. Formula la risposta finale spiegando l'esito chiaro (OK/APPROVATO, ESCALATION, RIFIUTATO) e le motivazioni dettagliate."""


memory = MemorySaver()
agent_app = Agent(
    model=llm,
    tools=tools_assicurativi,
    checkpointer=memory,
    system=system_prompt
)


# ==============================================================================
# 5. ESECUZIONE DELLE RICHIESTE
# ==============================================================================

def esegui_richiesta(testo_richiesta: str, thread_id: str = "default_thread"):
    config = {"configurable": {"thread_id": thread_id}}
    messages = [HumanMessage(content=testo_richiesta)]

    print(f"\n{'═'*60}")
    print(f"🚀 Richiesta: {testo_richiesta}")
    print(f"{'═'*60}")

    risposta_finale = ""
    chiedi_perito = False

    for event in agent_app.graph.stream({"messages": messages}, config):
        for node_val in event.values():
            # Controllo deterministico: se un tool ha restituito la stringa del perito
            # la riportiamo identica, senza dipendere da come la riformula il LLM.
            for m in node_val["messages"]:
                if isinstance(m, ToolMessage) and str(m.content).strip() == CHIEDI_PERITO:
                    chiedi_perito = True
            last_msg = node_val["messages"][-1]
            if isinstance(last_msg, HumanMessage):
                continue
            if not getattr(last_msg, "tool_calls", None) and last_msg.content:
                print(f"\n🤖 Risposta finale:\n{last_msg.content}")
                if isinstance(last_msg, AIMessage):
                    risposta_finale = last_msg.content

    # Valore restituito all'orchestrator: "chiedi a un perito" oppure la risposta finale
    return CHIEDI_PERITO if chiedi_perito else risposta_finale


# --- ESECUZIONE TEST SUI SINISTRI ---
if __name__ == "__main__":
    # Test 1: Sinistro CL001 (Valido e Approvabile)
    esegui_richiesta(
        "Ciao, puoi valutare la pratica per il sinistro CL001?",
        thread_id="pratica_CL001"
    )

    # Test 2: Sinistro CL002 (Documenti mancanti & Importo elevato)
    esegui_richiesta(
        "Analizza il sinistro CL002 e dimmi se possiamo approvarlo.",
        thread_id="pratica_CL002"
    )

    # Test 3: Sinistro CL003 (Polizza Disattiva)
    esegui_richiesta(
        "Qual è lo stato di approvazione per la pratica CL003?",
        thread_id="pratica_CL003"
    )



════════════════════════════════════════════════════════════
🚀 Richiesta: Ciao, puoi valutare la pratica per il sinistro CL001?
════════════════════════════════════════════════════════════
  🔧 Chiamata tool: get_info_sinistro({'claim_id': 'CL001'})

🤖 Risposta finale:
Dati Sinistro 'CL001':
- Policy ID: POL001
- Tipo Danno: collision
- Danno Stimato: 1200 EUR
- Documenti Inviati: ['photo', 'police_report']
  🔧 Chiamata tool: verifica_copertura_e_tipo_polizza({'policy_id': 'POL001', 'loss_type': 'collision'})

🤖 Risposta finale:
OK — Polizza attiva. Prodotto: 'Auto', Copertura: 'collision'.
  🔧 Chiamata tool: analizza_documenti({'claim_documents': ['photo', 'police_report'], 'loss_type': 'collision'})

🤖 Risposta finale:
OK — Documentazione completa. Documenti forniti: ['police_report', 'photo'].
  🔧 Chiamata tool: calcola_budget_e_limiti({'estimated_damage': 1200, 'policy_id': 'POL001', 'loss_type': 'collision'})

🤖 Risposta finale:
OK — Danno stimato (1200.0€) compreso tra franchigia

## Test di verifica trasparenza e correttezza dati estratti

In [4]:
# TEST DI VERIFICA TRASPARENZA E CORRETTEZZA DATI ESTRATTI

def verifica_correttezza_tool(claim_id_da_testare: str):
    print(f"\n============================================================")
    print(f" 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: {claim_id_da_testare}")
    print(f"============================================================")
    
    # 1. Recuperiamo il sinistro dal DB
    claim = CLAIMS_DB.get(claim_id_da_testare)
    if not claim:
        print(f"Errore: Il sinistro {claim_id_da_testare} non esiste nei dati!")
        return

    # Estrazione campi dal sinistro
    policy_id = claim["policy_id"]
    loss_type = claim["loss_type"]
    estimated_damage = claim["estimated_damage"]
    claim_docs = claim["documents"]

    # Recuperiamo i dati corrispondenti dagli altri file JSON
    policy = POLICIES_DB.get(policy_id, {})
    requirement = REQUIREMENTS_DB.get(loss_type, {})

    # STAMPA DEI DATI REALI CONTENUTI NEI JSON 
    print("\n 1. DATI GREZZI PRESENTI NEI FILE JSON:")
    print(f"   • Sinistro ({claim_id_da_testare}): Danno={estimated_damage}€, Documenti Inviati={claim_docs}")
    print(f"   • Polizza ({policy_id}): Attiva={policy.get('active')}, Prodotto={policy.get('product')}, Copertura={policy.get('coverage')}")
    print(f"   • Regole Polizza: Franchigia={policy.get('deductible')}€, Massimale={policy.get('max_claim')}€")
    print(f"   • Requisiti ({loss_type}): Documenti Richiesti={requirement.get('required_documents')}, Max Automatico={requirement.get('automatic_review_max')}€")

    # STAMPA DELLE RISPOSTE GENERATE DAI TOOL
    print("\n 2. RISPOSTE RESTITUITE DAI TUOI TOOL:")

    # Test Tool 1: Copertura e Tipo Polizza
    res_copertura = verifica_copertura_e_tipo_polizza.invoke({
        "policy_id": policy_id, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Copertura:   '{res_copertura}'")

    # Test Tool 2: Documenti
    res_docs = analizza_documenti.invoke({
        "claim_documents": claim_docs, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Documenti:   '{res_docs}'")

    # Test Tool 3: Budget e Limiti
    res_budget = calcola_budget_e_limiti.invoke({
        "estimated_damage": estimated_damage, 
        "policy_id": policy_id, 
        "loss_type": loss_type
    })
    print(f"   ► Tool Budget:      '{res_budget}'")

# --- ESEGUIAMO IL TEST SUI DIVERSI SINISTRI DEL VOSTRO PROGETTO ---
# Test 1: Sinistro CL001 (Caso perfetto)
verifica_correttezza_tool("CL001")

# Test 2: Sinistro CL002 (Caso con documenti mancanti e importo alto)
verifica_correttezza_tool("CL002")

# Test 3: Sinistro CL003 (Caso polizza disattiva)
verifica_correttezza_tool("CL003")


 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: CL001

 1. DATI GREZZI PRESENTI NEI FILE JSON:
   • Sinistro (CL001): Danno=1200€, Documenti Inviati=['photo', 'police_report']
   • Polizza (POL001): Attiva=True, Prodotto=Auto, Copertura=collision
   • Regole Polizza: Franchigia=250€, Massimale=15000€
   • Requisiti (collision): Documenti Richiesti=['photo', 'police_report'], Max Automatico=3000€

 2. RISPOSTE RESTITUITE DAI TUOI TOOL:
   ► Tool Copertura:   'OK — Polizza attiva. Prodotto: 'Auto', Copertura: 'collision'.'
   ► Tool Documenti:   'OK — Documentazione completa. Documenti forniti: ['photo', 'police_report'].'
   ► Tool Budget:      'OK — Danno stimato (1200.0€) compreso tra franchigia (250€) e massimale (15000€), ed entro la soglia automatica (3000€).'

 🔎 VERIFICA ESTRAZIONE E RISPOSTA PER IL SINISTRO: CL002

 1. DATI GREZZI PRESENTI NEI FILE JSON:
   • Sinistro (CL002): Danno=7000€, Documenti Inviati=['photo']
   • Polizza (POL002): Attiva=True, Prodotto=Home, Copertu